# Sync Orchestrator

Distributes skills from this library to configured AI tool environments
(Claude Code CLAUDE.md, GitHub Copilot instructions, etc.) using idempotent
boundary-marker injection.

**Workflow:** This notebook handles the _distribute_ side. To _pull_ upstream
updates first, run [`update-sourced-skills.ipynb`](update-sourced-skills.ipynb).

Destinations are defined in [`manifests/destinations.json`](manifests/destinations.json).
Only enabled destinations with assigned skills are processed.

## Phase 1 — Init

Load environment variables and manifests.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
# Ensure the repo root is correct by checking for manifests/
if not (REPO_ROOT / "manifests" / "origins.json").is_file():
    for p in [REPO_ROOT, *REPO_ROOT.parents]:
        if (p / "manifests" / "origins.json").is_file():
            REPO_ROOT = p
            break
    else:
        raise FileNotFoundError("manifests/origins.json not found")

sys.path.insert(0, str(REPO_ROOT))

from scripts.sync_engine import (
    load_env, load_origins, load_destinations,
    build_prompt_cache, distribute_all, format_report
)

env = load_env(REPO_ROOT)
origins = load_origins(REPO_ROOT)
destinations = load_destinations(REPO_ROOT)
CACHE_DIR = REPO_ROOT / ".cache"

print(f"Repo root: {REPO_ROOT}")
print(f"Origins:      {len(origins['skills'])} tracked + {len(origins.get('excluded', []))} excluded")
print(f"Destinations: {len(destinations['destinations'])} configured")

## Phase 2 — Fetch / Build Cache

Copy SKILL.md from each skill folder into `.cache/prompts/` so the
distribute phase has a flat directory of prompt files to work with.

In [ ]:
cached = build_prompt_cache(REPO_ROOT, origins)
print(f"Cached {len(cached)} skill prompt(s) to {CACHE_DIR / 'prompts'}")
for name in sorted(cached):
    print(f"  - {name}")

## Phase 3 — Distribute

Push cached prompts to each enabled destination. Uses idempotent
HTML-comment boundary markers so only the managed section is replaced;
any manual content outside the markers is preserved.

In [ ]:
results = distribute_all(destinations, CACHE_DIR, REPO_ROOT)
for r in results:
    icons = {'+': 'created', '~': 'updated', '=': 'unchanged',
             '-': 'skipped', '!': 'error'}
    icon = {v: k for k, v in icons.items()}.get(r['status'], '?')
    print(f"  [{icon}] {r['id']}: {r['status']} \u2014 {r.get('detail', '')}")

## Phase 4 — Report

Summary of everything that happened this run.

In [ ]:
print(format_report(None, results))